In [1]:
import os
#from azure.ai.ml import MLClient, Input, MpiDistribution, command
from azure.ai.ml import MLClient, Input, PyTorchDistribution, command
from azure.ai.ml.entities import AmlCompute, Environment, BuildContext, Data
from azure.identity import DefaultAzureCredential
from azure.ai.ml.constants import AssetTypes

# Azure ML workspace configuration
SUBSCRIPTION_ID = "8f94592c-747a-4bb7-ab3e-0b569993c33c"
RESOURCE_GROUP = "rg_amlworkspace"
WORKSPACE_NAME = "demo-mlworkspace01"
COMPUTE_CLUSTER = "demo-gpucluster01"

# authentication via managed identity or service principal (no hard-coded creds)
ml_client = MLClient(DefaultAzureCredential(), SUBSCRIPTION_ID, RESOURCE_GROUP, WORKSPACE_NAME)

# ensure compute cluster exists or create it
try:
    ml_client.compute.get(COMPUTE_CLUSTER)
except Exception:
    print("demo-gpucluster01 was not found")

#### Docker environment

In [2]:
CLOUD_DIR = "./cloud"

In [3]:
# job configuration
NUM_NODES = 2
NUM_GPU_PER_NODE = 1
#LOCAL_BATCH_SIZE = 8
#GLOBAL_BATCH_SIZE = 1024
#GRADIENT_ACCUMULATION_STEPS = GLOBAL_BATCH_SIZE // (LOCAL_BATCH_SIZE * NUM_GPU_PER_NODE * NUM_NODES)

# register AML inputs for data and model (register these assets in your workspace beforehand)
#wikipedia_dataset = Input(type="uri_file", path="azureml:wikipedia_dataset:1")
#train_data = Input(type="uri_file", path="azureml:ja_wiki_merged_train_0:1")
#model_dir = Input(type="uri_folder", path="azureml:llama2-7b-hf:1")

In [ ]:
# define distributed training job
#dist = MpiDistribution()
#dist.process_count_per_instance = NUM_GPU_PER_NODE
dist = PyTorchDistribution(
    process_count_per_instance=NUM_GPU_PER_NODE,
    node_count=NUM_NODES
)

job = command(
    code="./cloud",
    command=(
        "python megatron_lm/tools/preprocess_data.py \
        --input ${{inputs.train_data}}/wikidump.jsonl \
        --output-prefix ./outputs/wikidump \
        --tokenizer-type Llama2Tokenizer \
        --tokenizer-model ${{inputs.model_dir}} \
        --workers 1"
    ),
    inputs={
        "train_data": Input(
            type=AssetTypes.URI_FILE, 
            path="wiki_dump_02@latest"
        ),
        "model_dir": Input(
            type=AssetTypes.URI_FOLDER, 
            path="llama3-8b@latest"
        )
    },
    environment="llama3-8b-wiki_env@latest",
    compute=COMPUTE_CLUSTER,
    instance_count=NUM_NODES,
    distribution=dist,
    environment_variables={
        "LOGLEVEL": "INFO",
        "NCCL_DEBUG": "WARN",
        "NCCL_DEBUG_SUBSYS": "WARN",
        "PYTHONFAULTHANDLER": "1",
        "CUDA_LAUNCH_BLOCKING": "0"
    },
    display_name="llama3-8b-wiki-index",
    experiment_name="llama3-8b-wiki-index-exp"
)

In [5]:
# submit the job
returned_job = ml_client.jobs.create_or_update(job)
print(f"Job submitted: {returned_job.name}")
print(f"Monitor at: {returned_job.studio_url}")

Class AutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class AutoDeleteConditionSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseAutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class IntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class ProtectionLevelSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseIntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Uploading cloud (0.68 MBs): 100%|█████

Job submitted: bright_nutmeg_zsh8t9z80s
Monitor at: https://ml.azure.com/runs/bright_nutmeg_zsh8t9z80s?wsid=/subscriptions/8f94592c-747a-4bb7-ab3e-0b569993c33c/resourcegroups/rg_amlworkspace/workspaces/demo-mlworkspace01&tid=16b3c013-d300-468d-ac64-7eda0820b6d3
